In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

In [2]:
# Load data. Assuming files are in a 'data' directory.
X = np.load("data/x_train.npy")[:3000]
Y = np.load("data/y_train.npy")[:3000]

X_test = np.load("data/x_test.npy")
Y_test = np.load("data/y_test.npy")

# Add bias term to the input data X
X = np.hstack((X, np.ones((X.shape[0], 1)))) #[:1000]
#Y = Y[:1000]
X_test = np.hstack((X_test, np.ones((X_test.shape[0], 1))))

X = X + 0.00000000000000000001


print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

X shape: (3000, 785)
Y shape: (3000,)


In [3]:
def png_to_binary_array(image_path):
    """
    Converts a 28x28 PNG image to a numpy array of 0s and 1s, and adds a bias term.
    
    FIXED: The original function incorrectly appended '1' to the 2D array, leading to a shape mismatch.
    This version correctly flattens the 28x28 image (784 features) and then appends the bias term (785 features).
    """
    # Open the image and convert to grayscale
    img = Image.open(image_path).convert('L')  # 'L' mode = grayscale
    
    # Convert image to numpy array
    img_array = np.array(img)
    
    # Normalize to 0-1 (0 for black, 1 for non-black) based on user's intent
    binary_array = np.where(img_array > 0, 1, 0)
    
    # Flatten to 1D array (784 features)
    flattened_array = binary_array.flatten()
    
    # Add bias term (the '1' at the end) to make it 785 features
    flattened_array_with_bias = np.append(flattened_array, 1)
    
    return flattened_array_with_bias

In [4]:
test_0 = png_to_binary_array("data/png/binary_test_0.png")

test_1 = png_to_binary_array("data/png/binary_test_1.png")

In [5]:
def sigmoid(x):
    """
    Compute the sigmoid of x, where x can be a scalar, vector, or NumPy matrix.
    """
    out = 1 / (1 + np.exp(-x))
    
    return out

In [6]:
def sigmoid_prime(x):
    """
    Compute the derivative of the sigmoid function.
    """
    x  = sigmoid(x)
    out = np.multiply(x, (1 - x))
    return out

In [7]:
def calculate_outputs(theta, X, use_sigmoid = True):
    """
    Calculates the activations of one layer in the neural network.
    """
    print(X.shape, theta.shape)
    y_hat = X.dot(theta)
    
    if(use_sigmoid):
        out = sigmoid(y_hat)
    else:
        out = y_hat
    
    return out

In [8]:
def forward_prop(theta_layer1, theta_layer2, theta_layerOut, X, return_activation_array = False):
    """
    Uses the neural network to perform a forward propagation.
    """
    activations = []
    pre_activations = []

    print("layer 1")
    Z1 = calculate_outputs(theta_layer1, X, False)
    A1 = Z1 #sigmoid(Z1)
    print("layer 2")
    Z2 = calculate_outputs(theta_layer2, A1, False)
    A2 = Z2 #sigmoid(Z2)
    print("layer 3")
    Z3 = calculate_outputs(theta_layerOut, A2, False)
    A3 = Z3 #sigmoid(Z3)
    
    if(return_activation_array):
        pre_activations.append(Z1)
        activations.append(A1)
        pre_activations.append(Z2)
        activations.append(A2)
        pre_activations.append(Z3)
        activations.append(A3)
        return activations, pre_activations
    
    return A3

In [9]:
def calculate_gradient_output_layer(Y, A2, A3):
    """
    """
    n = X.shape[0]
    Y = Y.reshape((-1, 1))
    
    D3 = A3 - Y

    dW3 = ((A2.T.dot(D3)) / n)
    
    return dW3, D3

In [10]:
def calculate_gradient_hidden_layer(D3, W3, Z2, A1, n):
    """
    """
    
    dA2 = D3.dot(W3.T)

    s = Z2 #sigmoid_prime(Z2)

    D2 = np.multiply(dA2, s)

    gradients = (A1.T.dot(D2) / n)
    
    return gradients, D2

In [11]:
def calculate_mse(t1,t2,t3,X,Y):
    
    n = X.shape[0]
    
    y_hat = forward_prop(t1,t2,t3,X)


    bce = -np.mean(Y * np.log(y_hat) + (1 - Y) * np.log(1 - y_hat))
    
    return bce

In [12]:
LEARNING_RATE = 0.7
ITERATIONS = 100
FIRST_LAYER_NEURONS = 4
SECOND_LAYER_NEURONS = 4

m = X.shape[0]
i = X.shape[1]

# Initialize weights
theta_1 = np.random.rand(i, FIRST_LAYER_NEURONS).astype("float64")#*10
theta_2 = np.random.rand(FIRST_LAYER_NEURONS, SECOND_LAYER_NEURONS).astype("float64")#*10
theta_out = np.random.rand(SECOND_LAYER_NEURONS, 1).astype("float64")#*10

#theta_1 = np.zeros((i, FIRST_LAYER_NEURONS))
#theta_2 = np.zeros((FIRST_LAYER_NEURONS, SECOND_LAYER_NEURONS))
#theta_out = np.zeros((SECOND_LAYER_NEURONS, 1))

In [13]:
# Final Test
test_0 = png_to_binary_array("data/png/binary_test_0.png")
test_1 = png_to_binary_array("data/png/binary_test_1.png")

# Reshape for forward prop (1, 785)
test_0_input = test_0.reshape(1, -1)
test_1_input = test_1.reshape(1, -1)

In [14]:
def back_propagation_fixed(theta1, theta2, theta3, X, Y):
    #print(calculate_mse(theta_1, theta_2, theta_3, X, Y))

    #J_history = [calculate_mse(theta_1, theta_2, theta_3, X, Y)]
    
    n = X.shape[0]

    theta_1 = theta1
    theta_2 = theta2
    theta_3 = theta3
    
    for j in range(ITERATIONS):
        print(f"training iteration {j}")
        # 1. Forward Prop
        print(theta_1.shape, theta_2.shape, theta_3.shape, X.shape)
        activations, pre_activations = forward_prop(theta_1, theta_2, theta_3, X, True)
        
        # 2. Calculate Weight Gradients
        grad_3, D3 = calculate_gradient_output_layer(Y, activations[1], activations[2])
        
        grad_2, D2 = calculate_gradient_hidden_layer(D3, theta_3, pre_activations[1], activations[0], n)
        
        grad_1, D1 = calculate_gradient_hidden_layer(D2, theta_2, pre_activations[0], X, n)

        np.set_printoptions(precision=4, suppress=True)
        #print(grad_1, grad_2, grad_3)
        
        # 4. Update Weights (Stochastic Gradient Descent)
        theta_1 = theta_1 - LEARNING_RATE * grad_1
        theta_2 = theta_2 - LEARNING_RATE * grad_2
        theta_3 = theta_3 - LEARNING_RATE * grad_3
        
        #J_history.append(calculate_mse(theta_1, theta_2, theta_3, X, Y))
        
    return theta_1, theta_2, theta_3 #, np.array(J_history)

In [15]:
print(forward_prop(theta_1, theta_2, theta_out, X_test[:10]))
print(Y_test[:10])
print(theta_1, theta_2, theta_out)

layer 1
(10, 785) (785, 4)
layer 2
(10, 4) (4, 4)
layer 3
(10, 4) (4, 1)
[[39058.2827675 ]
 [59892.72356642]
 [18719.18473109]
 [73234.80731321]
 [39213.94707371]
 [26474.75627324]
 [42897.13982822]
 [41870.82342047]
 [62915.74326429]
 [64030.2924697 ]]
[7 2 1 0 4 1 4 9 5 9]
[[0.11663344 0.05287561 0.63336625 0.30949755]
 [0.42452883 0.02690369 0.87507864 0.21719458]
 [0.18513654 0.46871186 0.12316727 0.6665322 ]
 ...
 [0.35533541 0.16466601 0.643472   0.75369345]
 [0.54011815 0.1808534  0.21633479 0.67168858]
 [0.84292074 0.48468142 0.4334946  0.42778615]] [[0.93869548 0.59535355 0.91453863 0.63301203]
 [0.29498873 0.29272104 0.45207271 0.72722647]
 [0.75851793 0.40047262 0.11560764 0.66746938]
 [0.60807414 0.02099823 0.51921739 0.41120782]] [[0.64941183]
 [0.36863777]
 [0.52655356]
 [0.32198166]]


In [16]:
# Run the fixed training loop
theta_1_final, theta_2_final, theta_out_final = back_propagation_fixed(theta_1, theta_2, theta_out, X, Y)
#print(calculate_mse(theta_1_final, theta_2_final, theta_out_final, X, Y))


training iteration 0
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 1
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 2
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 3
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 4
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 5
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 6
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 7
(785, 4) (4, 4) (4, 1) (3000, 785)

/run/user/1000/ipykernel_5875/117744417.py:9: RuntimeWarning: overflow encountered in dot
  dW3 = ((A2.T.dot(D3)) / n)
/run/user/1000/ipykernel_5875/949592592.py:5: RuntimeWarning: overflow encountered in dot
  dA2 = D3.dot(W3.T)
/run/user/1000/ipykernel_5875/1647858097.py:29: RuntimeWarning: invalid value encountered in subtract
  theta_1 = theta_1 - LEARNING_RATE * grad_1
/run/user/1000/ipykernel_5875/1647858097.py:30: RuntimeWarning: invalid value encountered in subtract
  theta_2 = theta_2 - LEARNING_RATE * grad_2
/run/user/1000/ipykernel_5875/1647858097.py:31: RuntimeWarning: invalid value encountered in subtract
  theta_3 = theta_3 - LEARNING_RATE * grad_3


layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 8
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 9
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 10
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 11
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 12
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 13
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)
training iteration 14
(785, 4) (4, 4) (4, 1) (3000, 785)
layer 1
(3000, 785) (785, 4)
layer 2
(3000, 4) (4, 4)
layer 3
(3000, 4) (4, 1)


In [ ]:

# Plotting the cost history (optional)
plt.figure(figsize=(10, 6))
plt.plot(J_history_final[1:], label='MSE Cost', marker = "o")
plt.title('Mean Squared Error (MSE) Cost History')
plt.xlabel('Iteration (Epoch)')
plt.ylabel('Cost')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(forward_prop(theta_1_final, theta_2_final, theta_out_final, X_test[:10]))
print(Y_test[:10])
#print(f"Prediction for '1' (Expected close to 1): {forward_prop(theta_1_final, theta_2_final, theta_out_final, test_1_input)}")

In [ ]:
print(theta_1_final)
print(theta_2_final)
print(theta_out_final)